<a href="https://colab.research.google.com/github/DQN-Labs/Chess_AI/blob/main/llama_cppNEW.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title #Runtime Info
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)
from psutil import virtual_memory
ram_gb = virtual_memory().total / 1e9
print('Your runtime has {:.1f} gigabytes of available RAM\n'.format(ram_gb))
if ram_gb < 20:
  print('Not using a high-RAM runtime')
else:
  print('You are using a high-RAM runtime!')


In [ ]:
#@title ⚡ DQN Labs Ultra-Fast Setup (dqnGPT)

%cd /content/

# Clean previous stuff
!rm -rf build llama_bin.tar.gz *.gguf
print("removed old build")

# 1. Download prebuilt llama.cpp CUDA runtime
!wget -q https://huggingface.co/DQN-Labs/llamacpp-binaries-for-colab/resolve/main/llama_bin.tar.gz
print("got new build")
# Extract runtime
!tar -xzf llama_bin.tar.gz
print("extracted new build")
# 2. Download YOUR model
!wget -q https://huggingface.co/DQN-Labs/dqnGPT-v0.1-3.8B/resolve/main/dqnGPT-v0.1-3.8B.Q4_K_M.gguf
print("got gguf")
# 3. Verify
!ls build/bin | head -5

In [ ]:
!wget https://huggingface.co/DQN-Labs/dqnGPT-v0.1-3.8B/resolve/main/phi-3-mini-4k-instruct.Q4_K_M.gguf

In [ ]:
!export LD_LIBRARY_PATH=/content/build/bin:$LD_LIBRARY_PATH && \
./build/bin/llama-server \
    -m phi-3-mini-4k-instruct.Q4_K_M.gguf \
    -ngl 999 \
    -c 4096 \
    --port 8000 --no-webui

In [ ]:
!wget -q https://huggingface.co/api/models/DQN-Labs/dqnGPT-v0.1-3.8B -O model_info.json
!cat model_info.json | grep gguf

In [ ]:
!find /content -name "llama-server"

In [ ]:
#@title # Connect using the following address when the server is up.
from google.colab.output import eval_js
print(eval_js("google.colab.kernel.proxyPort(8000)"))

In [ ]:
#@title FastAPI



In [ ]:
#@title # inference
%cd /content/llama.cpp
!./server -m zephyr-7b-beta.Q6_K.gguf -ngl 9999 -c 0 --port 12345


In [ ]:
#@title CLOUDFLARED

!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

In [ ]:
import re

# Run cloudflared tunnel in background and get the public URL
cloudflared_proc = subprocess.Popen(
    ['./cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000', '--no-autoupdate'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = None
for line in cloudflared_proc.stdout:
    print(line.strip())
    match = re.search(r'(https://.*\.trycloudflare\.com)', line)
    if match:
        public_url = match.group(1)
        break

if public_url:
    print(f"\n✅ Public URL for Ollama:\n{public_url}")
else:
    raise RuntimeError("❌ Could not find public Cloudflare URL.")
